
# CV-STQA — M0 Baseline
### ResNet-50 + Temporal Mean Pooling + Regression

**Purpose:** Establish the first reproducible CV-STQA baseline.

```text
Video
  ↓
16 uniformly sampled frames
  ↓
224 × 224
  ↓
ImageNet-pretrained ResNet-50
  ↓
2048-D feature per frame
  ↓
Temporal mean pooling
  ↓
2048-D video feature
  ↓
MLP regression head
  ↓
6 VQA-MOS dimensions
```

### Kaggle Inputs
- **CV-STQA Configuration Split**: `train.csv`, `val.csv`, `test.csv`, `split_config.json`
- **MAC-VQA Videos**: actual videos
- **VQA-MOS**: supporting/original data if referenced by the CSVs

The notebook automatically searches `/kaggle/input`.


## 0. Imports

In [3]:
import os
import json
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr, kendalltau

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Device:", device)

PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
OpenCV: 4.14.0
CUDA available: True
GPU: Tesla T4
Device: cuda


## 1. Reproducibility

In [4]:

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)


Seed: 42


## 2. Find Phase-0 files automatically

In [5]:

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/m0_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def find_all(filename):
    return [Path(p) for p in glob.glob(str(INPUT_ROOT / "**" / filename), recursive=True)]

csv_candidates = {
    "train": find_all("train.csv"),
    "val": find_all("val.csv"),
    "test": find_all("test.csv"),
}
json_candidates = find_all("split_config.json")

print("CSV candidates:")
for k, paths in csv_candidates.items():
    print(f"  {k}:")
    for p in paths:
        print("   ", p)

print("\nsplit_config.json candidates:")
for p in json_candidates:
    print("  ", p)

if any(len(v) == 0 for v in csv_candidates.values()):
    raise FileNotFoundError("Could not find train.csv, val.csv, and/or test.csv under /kaggle/input.")
if len(json_candidates) == 0:
    raise FileNotFoundError("Could not find split_config.json under /kaggle/input.")

def prefer_configuration_path(paths):
    preferred = [
        p for p in paths
        if "configuration" in str(p).lower() or "split" in str(p).lower()
    ]
    return preferred[0] if preferred else paths[0]

TRAIN_CSV = prefer_configuration_path(csv_candidates["train"])
VAL_CSV = prefer_configuration_path(csv_candidates["val"])
TEST_CSV = prefer_configuration_path(csv_candidates["test"])
SPLIT_JSON = json_candidates[0]

print("\nSelected:")
print("TRAIN:", TRAIN_CSV)
print("VAL  :", VAL_CSV)
print("TEST :", TEST_CSV)
print("JSON :", SPLIT_JSON)


CSV candidates:
  train:
    /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/train.csv
  val:
    /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/val.csv
  test:
    /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/test.csv

split_config.json candidates:
   /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/split_config.json

Selected:
TRAIN: /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/train.csv
VAL  : /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/val.csv
TEST : /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/test.csv
JSON : /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/split_config.json


## 3. Inspect the split files

In [6]:

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("TRAIN:", train_df.shape)
print("VAL  :", val_df.shape)
print("TEST :", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nFirst 3 train rows:")
display(train_df.head(3))

with open(SPLIT_JSON, "r") as f:
    split_config = json.load(f)

print("\nSplit configuration:")
print(json.dumps(split_config, indent=2)[:5000])


TRAIN: (700, 8)
VAL  : (150, 8)
TEST : (150, 8)

Train columns:
['video_id', 'TSV', 'B', 'SR', 'S', 'U', 'O', 'video_path']

First 3 train rows:


,video_id,TSV,B,SR,S,U,O,video_path
0,68-11.mp4,2.0025,2.1235,1.8600,1.72120,1.8126,3.5140,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...
1,48-3.mp4,2.1210,3.0000,1.6898,2.30670,2.3584,4.1275,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...
2,2-10.mp4,1.4000,1.6566,1.3648,0.68874,1.4057,2.9560,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...



Split configuration:
{
  "protocol": "A_random",
  "seed": 42,
  "train_ratio": 0.7,
  "val_ratio": 0.15,
  "test_ratio": 0.15,
  "num_frames": 16,
  "grid_size": 7,
  "image_size": 224
}


## 4. Identify video column and MOS targets

In [8]:
# Identify video column and the 6 MOS target columns

VIDEO_COL = "video_id"

TARGET_COLS = {
    "TSV": "TSV",
    "Brightness": "B",
    "Specular": "SR",
    "Stability": "S",
    "Utility": "U",
    "Overall": "O"
}

# Verify all required columns exist
required_columns = [
    VIDEO_COL,
    "TSV",
    "B",
    "SR",
    "S",
    "U",
    "O"
]

missing_columns = [
    col
    for col in required_columns
    if col not in train_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {train_df.columns.tolist()}"
    )

TARGET_NAMES = list(TARGET_COLS.keys())

TARGET_COLUMNS = [
    TARGET_COLS[name]
    for name in TARGET_NAMES
]

print("Video column:", VIDEO_COL)

print("\nTarget mapping:")

for i, (name, col) in enumerate(
    TARGET_COLS.items(),
    start=1
):
    print(
        f"{i}. {name} -> CSV column '{col}'"
    )

print("\nAll 6 target columns found successfully.")

Video column: video_id

Target mapping:
1. TSV -> CSV column 'TSV'
2. Brightness -> CSV column 'B'
3. Specular -> CSV column 'SR'
4. Stability -> CSV column 'S'
5. Utility -> CSV column 'U'
6. Overall -> CSV column 'O'

All 6 target columns found successfully.


## 5. Locate all videos and match them to the CSVs

In [9]:

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpg", ".mpeg"}

all_video_paths = []
for ext in VIDEO_EXTENSIONS:
    all_video_paths.extend(INPUT_ROOT.glob(f"**/*{ext}"))
    all_video_paths.extend(INPUT_ROOT.glob(f"**/*{ext.upper()}"))

all_video_paths = sorted(set(p.resolve() for p in all_video_paths))

print("Video files found:", len(all_video_paths))

if not all_video_paths:
    raise FileNotFoundError(
        "No video files found under /kaggle/input. "
        "Make sure MAC-VQA Videos is attached."
    )

video_lookup = {}
for p in all_video_paths:
    for key in {
        p.name, p.stem, p.name.lower(), p.stem.lower(),
        str(p).lower()
    }:
        video_lookup.setdefault(key, p)

def resolve_video_path(value):
    value = str(value).strip()
    keys = [
        value,
        Path(value).name,
        Path(value).stem,
        value.lower(),
        Path(value).name.lower(),
        Path(value).stem.lower()
    ]
    for key in keys:
        if key in video_lookup:
            return str(video_lookup[key])

    norm = Path(value).stem.lower()
    matches = [p for p in all_video_paths if p.stem.lower() == norm]
    return str(matches[0]) if len(matches) == 1 else None

for df in [train_df, val_df, test_df]:
    df["video_path"] = df[VIDEO_COL].map(resolve_video_path)

for name, df in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
    missing = int(df["video_path"].isna().sum())
    print(f"{name}: {len(df)} rows | missing videos: {missing}")
    if missing:
        print(df.loc[df["video_path"].isna(), VIDEO_COL].head(10).tolist())

total_missing = sum(
    int(df["video_path"].isna().sum())
    for df in [train_df, val_df, test_df]
)

if total_missing:
    raise FileNotFoundError(
        f"{total_missing} CSV entries could not be matched to video files."
    )

print("\nAll CSV video entries successfully matched.")


Video files found: 1000
TRAIN: 700 rows | missing videos: 0
VAL: 150 rows | missing videos: 0
TEST: 150 rows | missing videos: 0

All CSV video entries successfully matched.


## 6. M0 configuration

In [10]:

NUM_FRAMES = 16
IMAGE_SIZE = 224
FRAME_BATCH_SIZE = 32
FEATURE_DIM = 2048
NUM_TARGETS = 6

FEATURE_DIR = OUTPUT_ROOT / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("Frames/video:", NUM_FRAMES)
print("Image size:", IMAGE_SIZE)
print("Feature dimension:", FEATURE_DIM)


Frames/video: 16
Image size: 224
Feature dimension: 2048


## 7. Load ImageNet-pretrained ResNet-50

In [12]:

weights = ResNet50_Weights.IMAGENET1K_V2
resnet = resnet50(weights=weights)

# Remove the ImageNet classification layer.
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

preprocess = weights.transforms()

print("ResNet-50 loaded.")
print("Frame feature dimension: 2048")


ResNet-50 loaded.
Frame feature dimension: 2048


## 8. Uniform frame sampling + feature extraction

In [13]:
def sample_frame_indices(num_video_frames, num_samples=16):

    if num_video_frames <= 0:
        raise ValueError("Video contains no frames.")

    if num_video_frames == 1:
        return np.zeros(
            num_samples,
            dtype=np.int64
        )

    indices = np.linspace(
        0,
        num_video_frames - 1,
        num_samples
    )

    return np.round(
        indices
    ).astype(np.int64)


def load_sampled_frames(
    video_path,
    num_samples=16
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames <= 0:
        cap.release()
        raise RuntimeError(
            f"Could not determine frame count: {video_path}"
        )

    indices = sample_frame_indices(
        total_frames,
        num_samples
    )

    frames = []

    for idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(idx)
        )

        success, frame = cap.read()

        if not success:
            cap.release()
            raise RuntimeError(
                f"Could not read frame {idx} "
                f"from {video_path}"
            )

        # OpenCV BGR -> RGB
        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # HWC -> CHW
        frame = torch.from_numpy(
            frame
        ).permute(2, 0, 1)

        # ResNet-50 preprocessing
        frame = preprocess(frame)

        frames.append(frame)

    cap.release()

    return torch.stack(frames)


@torch.inference_mode()
def extract_video_feature(video_path):

    frames = load_sampled_frames(
        video_path,
        NUM_FRAMES
    )

    frame_features = []

    for start in range(
        0,
        len(frames),
        FRAME_BATCH_SIZE
    ):

        batch = frames[
            start:start + FRAME_BATCH_SIZE
        ].to(
            device,
            non_blocking=True
        )

        features = feature_extractor(
            batch
        ).flatten(1)

        frame_features.append(
            features.cpu()
        )

    frame_features = torch.cat(
        frame_features,
        dim=0
    )

    # Temporal mean pooling
    # [16, 2048] -> [2048]
    video_feature = frame_features.mean(
        dim=0
    )

    return video_feature.numpy().astype(
        np.float32
    )

## 9. Extract and cache ResNet-50 video features

In [14]:

def extract_split_features(df, split_name):
    feature_file = FEATURE_DIR / f"{split_name}_resnet50_mean.npy"
    id_file = FEATURE_DIR / f"{split_name}_ids.csv"

    if feature_file.exists() and id_file.exists():
        print(f"{split_name}: cached features found — skipping extraction.")
        return np.load(feature_file), pd.read_csv(id_file)

    features = []
    print(f"Extracting {split_name} features for {len(df)} videos...")

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=split_name
    ):
        features.append(extract_video_feature(row["video_path"]))

    features = np.stack(features)

    ids = df[[VIDEO_COL, "video_path"]].copy()
    np.save(feature_file, features)
    ids.to_csv(id_file, index=False)

    print("Saved:", feature_file)
    print("Shape:", features.shape)
    return features, ids

train_features, train_ids = extract_split_features(train_df, "train")
val_features, val_ids = extract_split_features(val_df, "val")
test_features, test_ids = extract_split_features(test_df, "test")

print("\nFeature shapes:")
print("Train:", train_features.shape)
print("Val  :", val_features.shape)
print("Test :", test_features.shape)


Extracting train features for 700 videos...


train:   0%|          | 0/700 [00:00<?, ?it/s]

Saved: /kaggle/working/m0_outputs/features/train_resnet50_mean.npy
Shape: (700, 2048)
Extracting val features for 150 videos...


val:   0%|          | 0/150 [00:00<?, ?it/s]

Saved: /kaggle/working/m0_outputs/features/val_resnet50_mean.npy
Shape: (150, 2048)
Extracting test features for 150 videos...


test:   0%|          | 0/150 [00:00<?, ?it/s]

Saved: /kaggle/working/m0_outputs/features/test_resnet50_mean.npy
Shape: (150, 2048)

Feature shapes:
Train: (700, 2048)
Val  : (150, 2048)
Test : (150, 2048)


## 10. Prepare MOS targets

In [15]:

def get_targets(df):
    y = df[TARGET_COLUMNS].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)
    if np.isnan(y).any():
        raise ValueError("NaN/non-numeric values found in MOS target columns.")
    return y

y_train = get_targets(train_df)
y_val = get_targets(val_df)
y_test = get_targets(test_df)

print("Target names:", TARGET_NAMES)
print("Train targets:", y_train.shape)
print("Val targets  :", y_val.shape)
print("Test targets :", y_test.shape)


Target names: ['TSV', 'Brightness', 'Specular', 'Stability', 'Utility', 'Overall']
Train targets: (700, 6)
Val targets  : (150, 6)
Test targets : (150, 6)


## 11. M0 regression head

In [16]:

class M0Regressor(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512, output_dim=6):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.regressor(x)

model = M0Regressor(
    input_dim=FEATURE_DIM,
    hidden_dim=512,
    output_dim=NUM_TARGETS
).to(device)

print(model)


M0Regressor(
  (regressor): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=6, bias=True)
  )
)


## 12. Training configuration

In [17]:

class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    FeatureDataset(train_features, y_train),
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    FeatureDataset(val_features, y_val),
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)

EPOCHS = 50
PATIENCE = 10

print("Max epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)


Max epochs: 50
Early stopping patience: 10


## 13. Train M0

In [18]:

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for X, y in train_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_loss = float(np.mean(train_losses))

    model.eval()
    val_losses = []

    with torch.inference_mode():
        for X, y in val_loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = model(X)
            val_losses.append(criterion(pred, y).item())

    val_loss = float(np.mean(val_losses))
    scheduler.step(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": optimizer.param_groups[0]["lr"]
    })

    print(
        f"Epoch {epoch:03d} | "
        f"Train MSE: {train_loss:.5f} | "
        f"Val MSE: {val_loss:.5f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_ROOT / "training_history.csv", index=False)

print("Best validation MSE:", best_val_loss)


Epoch 001 | Train MSE: 1.54473 | Val MSE: 0.46869 | LR: 1.00e-03
Epoch 002 | Train MSE: 0.47813 | Val MSE: 0.30259 | LR: 1.00e-03
Epoch 003 | Train MSE: 0.31832 | Val MSE: 0.25069 | LR: 1.00e-03
Epoch 004 | Train MSE: 0.25630 | Val MSE: 0.22348 | LR: 1.00e-03
Epoch 005 | Train MSE: 0.23825 | Val MSE: 0.20918 | LR: 1.00e-03
Epoch 006 | Train MSE: 0.20851 | Val MSE: 0.18578 | LR: 1.00e-03
Epoch 007 | Train MSE: 0.19660 | Val MSE: 0.17661 | LR: 1.00e-03
Epoch 008 | Train MSE: 0.18498 | Val MSE: 0.16611 | LR: 1.00e-03
Epoch 009 | Train MSE: 0.17463 | Val MSE: 0.16552 | LR: 1.00e-03
Epoch 010 | Train MSE: 0.17099 | Val MSE: 0.15169 | LR: 1.00e-03
Epoch 011 | Train MSE: 0.16169 | Val MSE: 0.15009 | LR: 1.00e-03
Epoch 012 | Train MSE: 0.15413 | Val MSE: 0.14689 | LR: 1.00e-03
Epoch 013 | Train MSE: 0.14642 | Val MSE: 0.14889 | LR: 1.00e-03
Epoch 014 | Train MSE: 0.14152 | Val MSE: 0.14530 | LR: 1.00e-03
Epoch 015 | Train MSE: 0.14301 | Val MSE: 0.15163 | LR: 1.00e-03
Epoch 016 | Train MSE: 0.

## 14. Generate predictions

In [19]:

@torch.inference_mode()
def predict_features(model, features):
    model.eval()
    X = torch.tensor(features, dtype=torch.float32)
    loader = DataLoader(X, batch_size=256, shuffle=False)

    outputs = []
    for batch in loader:
        outputs.append(model(batch.to(device)).cpu().numpy())

    return np.concatenate(outputs, axis=0)

train_pred = predict_features(model, train_features)
val_pred = predict_features(model, val_features)
test_pred = predict_features(model, test_features)

print("Train:", train_pred.shape)
print("Val  :", val_pred.shape)
print("Test :", test_pred.shape)


Train: (700, 6)
Val  : (150, 6)
Test : (150, 6)


## 15. Evaluate M0

In [20]:

def safe_corr(metric_fn, y_true, y_pred):
    try:
        result = metric_fn(y_true, y_pred)
        return float(result.statistic if hasattr(result, "statistic") else result)
    except Exception:
        return np.nan

def evaluate_predictions(y_true, y_pred, target_names):
    rows = []

    for i, name in enumerate(target_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        rows.append({
            "Dimension": name,
            "PLCC": safe_corr(pearsonr, yt, yp),
            "SRCC": safe_corr(spearmanr, yt, yp),
            "KRCC": safe_corr(kendalltau, yt, yp),
            "RMSE": float(np.sqrt(mean_squared_error(yt, yp)))
        })

    result = pd.DataFrame(rows)

    result.loc[len(result)] = {
        "Dimension": "Mean",
        "PLCC": result["PLCC"].mean(),
        "SRCC": result["SRCC"].mean(),
        "KRCC": result["KRCC"].mean(),
        "RMSE": result["RMSE"].mean()
    }
    return result

val_results = evaluate_predictions(y_val, val_pred, TARGET_NAMES)
test_results = evaluate_predictions(y_test, test_pred, TARGET_NAMES)

print("========== VALIDATION ==========")
display(val_results)

print("\n========== TEST ==========")
display(test_results)


========== VALIDATION ==========


,Dimension,PLCC,SRCC,KRCC,RMSE
0,TSV,0.923142,0.904554,0.737300,0.271838
1,Brightness,0.775215,0.734180,0.550779,0.495458
2,Specular,0.828688,0.809054,0.627009,0.383325
3,Stability,0.931950,0.920281,0.757852,0.293379
4,Utility,0.895140,0.897634,0.730679,0.335236
5,Overall,0.937087,0.932025,0.775618,0.369473
6,Mean,0.881871,0.866288,0.696540,0.358118



========== TEST ==========


,Dimension,PLCC,SRCC,KRCC,RMSE
0,TSV,0.907743,0.899049,0.739122,0.296326
1,Brightness,0.798600,0.755138,0.575389,0.491315
2,Specular,0.856293,0.857654,0.671948,0.347537
3,Stability,0.922511,0.921806,0.755168,0.301744
4,Utility,0.891618,0.897439,0.713870,0.330193
5,Overall,0.925974,0.932125,0.770753,0.397656
6,Mean,0.883790,0.877202,0.704375,0.360795


## 16. Save everything needed for M1 and the paper

In [21]:

model_path = OUTPUT_ROOT / "M0_resnet50_temporal_mean_regressor.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "feature_dim": FEATURE_DIM,
    "num_frames": NUM_FRAMES,
    "target_names": TARGET_NAMES,
    "seed": SEED,
    "best_val_mse": best_val_loss,
}, model_path)

for split_name, df, y_true, y_pred in [
    ("val", val_df, y_val, val_pred),
    ("test", test_df, y_test, test_pred),
]:
    pred_df = df[[VIDEO_COL]].copy()

    for i, name in enumerate(TARGET_NAMES):
        pred_df[f"{name}_true"] = y_true[:, i]
        pred_df[f"{name}_pred"] = y_pred[:, i]

    pred_df.to_csv(
        OUTPUT_ROOT / f"{split_name}_predictions.csv",
        index=False
    )

val_results.to_csv(OUTPUT_ROOT / "val_metrics.csv", index=False)
test_results.to_csv(OUTPUT_ROOT / "test_metrics.csv", index=False)

m0_config = {
    "model": "ResNet-50 ImageNet pretrained + temporal mean pooling + MLP regression",
    "num_frames": NUM_FRAMES,
    "feature_dim": FEATURE_DIM,
    "image_size": IMAGE_SIZE,
    "hidden_dim": 512,
    "dropout": 0.30,
    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs_max": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "targets": TARGET_NAMES,
}

with open(OUTPUT_ROOT / "M0_config.json", "w") as f:
    json.dump(m0_config, f, indent=2)

print("M0 outputs saved to:", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUT_ROOT))


M0 outputs saved to: /kaggle/working/m0_outputs
  M0_config.json
  M0_resnet50_temporal_mean_regressor.pt
  features/test_ids.csv
  features/test_resnet50_mean.npy
  features/train_ids.csv
  features/train_resnet50_mean.npy
  features/val_ids.csv
  features/val_resnet50_mean.npy
  test_metrics.csv
  test_predictions.csv
  training_history.csv
  val_metrics.csv
  val_predictions.csv



# M0 COMPLETE

### Keep these for M1

```text
m0_outputs/
├── features/
│   ├── train_resnet50_mean.npy
│   ├── val_resnet50_mean.npy
│   └── test_resnet50_mean.npy
├── M0_resnet50_temporal_mean_regressor.pt
├── val_predictions.csv
├── test_predictions.csv
├── val_metrics.csv
├── test_metrics.csv
├── training_history.csv
└── M0_config.json
```

**Most important:** keep the `features/*.npy`, `test_metrics.csv`, `test_predictions.csv`, and `M0_config.json`.

Do not tune M0 using the test set.
